# Генератор рукописных строк (EN + RU) — примеры

Шрифты в `assets/fonts_ru` / `assets/fonts_en` (`merge_fonts.py` или `fetch_fonts.py`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.synth import HandwrittenLineGenerator, make_generator

RU_FONTS = str(ROOT / 'assets' / 'fonts_ru')
EN_FONTS = str(ROOT / 'assets' / 'fonts_en')

## Сэмплим пары (текст, картинка)

Строки берутся только из реального текста. `*_text_weights` (длина == числу папок) задают,
из какой папки брать чаще; `[]` = по числу файлов. Пустые `*_text_dirs` -> аварийный встроенный словарь.
Иногда вносятся ошибки школьного типа (о->а, пропуск пунктуации) — они идут и в картинку, и в таргет.

In [ ]:
RU_TEXTS = ['/data/ru_big', '/data/ru_small']   # <- твои папки с русскими .txt
EN_TEXTS = ['/data/en_texts']                   # <- английские .txt

gen = HandwrittenLineGenerator.from_dirs(
    ru_text_dirs=RU_TEXTS, en_text_dirs=EN_TEXTS,
    ru_text_weights=[0.5, 0.5],   # из каждой ru-папки одинаково часто ([] = по числу файлов)
    ru_font_dirs=RU_FONTS, en_font_dirs=EN_FONTS,
    p_ru=0.5, len_chars=(15, 45), p_hyphenate=0.2, curriculum=False,
)
print('файлов: ru=%d en=%d' % (gen.sampler.n_files('ru'), gen.sampler.n_files('en')))

for i in range(6):
    img, text = gen.sample(make_generator(42, 0, i))
    print(text)
    display(img)

## В обучении (на лету)

Бесконечный поток -> `IterableDataset` с пер-воркерным сидом (см. `src/data.py`).

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader

class SynthLines(IterableDataset):
    def __init__(self, gen, base_seed=42):
        self.gen, self.base_seed = gen, base_seed
    def __iter__(self):
        info = torch.utils.data.get_worker_info(); wid = info.id if info else 0; i = 0
        while True:
            img, text = self.gen.sample(make_generator(self.base_seed, wid, i), step=i)
            yield {'image': img, 'text': text}; i += 1

loader = DataLoader(SynthLines(gen), batch_size=4, num_workers=0,
                    collate_fn=lambda b: ([x['image'] for x in b], [x['text'] for x in b]))
images, texts = next(iter(loader))
print(texts)
# дальше: processor(images=images).pixel_values + tokenizer(texts) -> TrOCR

Подсказки: `p_ru` — доля русских строк; `*_text_weights` — частота папок; `len_chars` — длина строки;
блок ошибок: `p_text_error`/`p_letter_sub`/`p_drop_punct`/`p_typo`; `step` в `sample(rng, step)` — сложность.